## 1) PRE-DEMO SETUP

## Install Dependencies

First, install the required packages. This only needs to be run once per session.

In [ ]:
# Install neo4j-graphrag with Bedrock support
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git" python-dotenv pydantic-settings nest-asyncio -q

## Setup

Import required modules and configure the environment.

In [ ]:
from data_utils import Neo4jConnection, DataLoader, split_text, get_embedder, get_llm
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

## 2) Load up some sample data

## Sample Data

We'll load text from `company_data.txt` representing content from an SEC 10-K filing.

> **Note:** In production, you would use `pypdf` or similar libraries to extract text from PDF files. We use a pre-defined text file here for fast, reproducible results.

In [ ]:
# Load text from file using DataLoader
loader = DataLoader("company_data.txt")
SAMPLE_TEXT = loader.text

# Document metadata
DOCUMENT_PATH = "form10k-sample/apple-2023-10k.pdf"
DOCUMENT_PAGE = 1

metadata = loader.get_metadata()
print(f"Loaded from: {metadata['name']}")
print(f"Sample text length: {metadata['size']} characters")
print(f"\n{SAMPLE_TEXT}")

## Connect to Neo4j

Create a connection to your Neo4j database using the `Neo4jConnection` utility class.

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

## Clear Existing Data (Optional)

For a clean start, remove any existing Document and Chunk nodes from previous runs using the utility method.

In [ ]:
neo4j.clear_graph()

## Create Document Node

First, create a Document node to represent the source file. This node stores metadata about where the content came from, which is essential for:

- **Provenance tracking**: Knowing which file a chunk originated from
- **Filtering**: Searching only within specific documents
- **Citations**: Providing references in generated answers

We use `elementId(d)` to get the node's internal identifier, which we'll use to link chunks to this document.

In [ ]:
def create_document(driver, path: str, page: int) -> str:
    """Create a Document node and return its element ID."""
    with driver.session() as session:
        result = session.run("""
            CREATE (d:Document {path: $path, page: $page})
            RETURN elementId(d) as doc_id
        """, path=path, page=page)
        return result.single()["doc_id"]

doc_id = create_document(driver, DOCUMENT_PATH, DOCUMENT_PAGE)
print(f"Created Document node with ID: {doc_id}")

## Split Text with FixedSizeSplitter

The `FixedSizeSplitter` from neo4j-graphrag automatically splits text into chunks of a specified size with overlap.

- **chunk_size**: Maximum characters per chunk
- **chunk_overlap**: Characters to overlap between chunks (preserves context)

In [ ]:
# Split text using the utility function (smaller chunks for demo)
chunks_text = split_text(SAMPLE_TEXT, chunk_size=400, chunk_overlap=50)

print(f"Split into {len(chunks_text)} chunks:\n")
for i, chunk in enumerate(chunks_text):
    print(f"Chunk {i}: {len(chunk)} chars")
    print(f"  {chunk}\n")

## Create Chunk Nodes

Create Chunk nodes for each piece of text and link them to the Document with `FROM_DOCUMENT` relationships.

Each Chunk node stores:
- `text`: The actual chunk content
- `index`: The position in the original document (0, 1, 2, ...)

The `FROM_DOCUMENT` relationship points from the Chunk back to its source Document, enabling queries like "find all chunks from document X" or "which document does this chunk belong to?"

In [ ]:
def create_chunks(driver, doc_id: str, chunks: list[str]) -> list[str]:
    """Create Chunk nodes linked to a Document. Returns chunk element IDs."""
    chunk_ids = []
    with driver.session() as session:
        for index, text in enumerate(chunks):
            result = session.run("""
                MATCH (d:Document) WHERE elementId(d) = $doc_id
                CREATE (c:Chunk {text: $text, index: $index})
                CREATE (c)-[:FROM_DOCUMENT]->(d)
                RETURN elementId(c) as chunk_id
            """, doc_id=doc_id, text=text, index=index)
            chunk_id = result.single()["chunk_id"]
            chunk_ids.append(chunk_id)
            print(f"Created Chunk {index}")
    return chunk_ids

chunk_ids = create_chunks(driver, doc_id, chunks_text)
print(f"\nCreated {len(chunk_ids)} chunks")

## Link Chunks with NEXT_CHUNK

Create `NEXT_CHUNK` relationships between sequential chunks. This preserves the original document order and enables a powerful retrieval pattern: **context expansion**.

When you find a relevant chunk via vector search, the `NEXT_CHUNK` relationships let you easily retrieve:
- The **previous chunk** for background context
- The **next chunk** for continuation and conclusions

This is one of the key advantages of storing chunks in a graph rather than a simple vector store.

In [ ]:
def link_chunks(driver, chunk_ids: list[str]):
    """Create NEXT_CHUNK relationships between sequential chunks."""
    with driver.session() as session:
        for i in range(len(chunk_ids) - 1):
            session.run("""
                MATCH (c1:Chunk) WHERE elementId(c1) = $id1
                MATCH (c2:Chunk) WHERE elementId(c2) = $id2
                CREATE (c1)-[:NEXT_CHUNK]->(c2)
            """, id1=chunk_ids[i], id2=chunk_ids[i+1])
        print(f"Created {len(chunk_ids) - 1} NEXT_CHUNK relationships")

link_chunks(driver, chunk_ids)

## Summary

In this notebook, you learned the foundational graph structure for GraphRAG applications:

1. **Document-Chunk structure** - Documents are split into smaller chunks for efficient retrieval. Each chunk is small enough to embed and retrieve precisely.

2. **FROM_DOCUMENT relationship** - Links chunks back to their source document, enabling provenance tracking and document-level filtering.

3. **NEXT_CHUNK relationship** - Preserves the sequential order of chunks, enabling context expansion during retrieval.

This basic structure is the foundation for all GraphRAG applications. The graph structure gives us capabilities that simple vector stores don't have - namely, the ability to traverse relationships to gather additional context.

In the next notebook, you'll learn to add **embeddings** to these chunks, enabling semantic similarity search.

---

**Next:** [Embeddings and Vector Search](02_embeddings.ipynb)

# Embeddings and Vector Search

This notebook demonstrates how to generate embeddings for text chunks and perform vector similarity search using Neo4j. Embeddings are the foundation of semantic search - the ability to find content by meaning rather than exact keyword matches.

**Prerequisites:** Review [01 Data Loading](01_data_loading.ipynb) to understand the Document-Chunk graph structure. This notebook is self-contained and will create its own data.

**Learning Objectives:**
- Understand what embeddings are and why they matter for GraphRAG
- Use `FixedSizeSplitter` to automatically chunk text
- Generate embeddings using AWS Bedrock (Amazon Titan)
- Create a vector index in Neo4j
- Perform similarity search to find relevant chunks

---

## What are Embeddings?

Embeddings are numerical representations (vectors) of text that capture semantic meaning. The key insight is that **similar texts produce similar vectors**, enabling semantic search.

```
"Apple makes iPhones" → [0.12, -0.45, 0.78, ...] (1024 dimensions)
"The company produces smartphones" → [0.11, -0.44, 0.77, ...] (similar vector!)
```

This is powerful because:
- A search for "smartphone manufacturer" will find content about "Apple makes iPhones" even though none of those exact words appear
- The embedding model understands that "produces" and "makes" are semantically similar
- You don't need to anticipate every possible way a user might phrase their query

**How similarity is measured:** We use **cosine similarity** to compare vectors. A score of 1.0 means identical direction (very similar), while 0.0 means perpendicular (unrelated). In practice, scores above 0.8 typically indicate strong semantic similarity.

## Clear Existing Data

Remove any existing Document and Chunk nodes from previous runs.

In [ ]:
neo4j.clear_graph()

## Initialize LLM and Embedder

Create an embedder using AWS Bedrock. The embedding model is configured in `CONFIG.txt` via the `EMBEDDING_MODEL_ID` setting.

We're using **Amazon Titan Text Embeddings V2**, which produces 1024-dimensional vectors. This is important because:
- The vector index dimensions must match the embedder output
- All chunks and queries must use the same embedding model for meaningful comparisons

In [ ]:
llm = get_llm()
embedder = get_embedder()
print(f"Embedder initialized: {embedder.model_id}")

## Generate Embeddings

Generate an embedding vector for each chunk. This calls the AWS Bedrock embedding API.

Each call to `embed_query()`:
1. Sends the text to Amazon Titan via AWS Bedrock
2. Returns a list of 1024 floating-point numbers
3. These numbers encode the semantic meaning of the text

> **Note:** Embedding generation has a cost (typically fractions of a cent per call), but it's a one-time operation. Once stored, embeddings can be searched repeatedly without additional API calls.

In [ ]:
# Generate embeddings for each chunk
chunk_embeddings = []
for i, text in enumerate(chunks_text):
    embedding = embedder.embed_query(text)
    chunk_embeddings.append({
        "text": text,
        "index": i,
        "embedding": embedding
    })
    print(f"Chunk {i}: Generated {len(embedding)}-dimensional embedding")

print(f"\nFirst 5 values of chunk 0's embedding: {chunk_embeddings[0]['embedding'][:5]}")

## Store in Neo4j with Embeddings

Create Document and Chunk nodes, storing the embedding vector on each Chunk as a property.

Neo4j can store vectors (lists of floats) as node properties. This is efficient because:
- The embedding is stored directly with the chunk text it represents
- No need for a separate vector database
- Graph traversals can access both text and embeddings seamlessly

In [ ]:
def store_chunks_with_embeddings(driver, doc_path: str, chunk_data: list[dict]):
    """Store Document and Chunk nodes with embeddings."""
    with driver.session() as session:
        # Create Document
        session.run("""
            CREATE (d:Document {path: $path})
        """, path=doc_path)
        print(f"Created Document: {doc_path}")
        
        # Create Chunks with embeddings
        for chunk in chunk_data:
            session.run("""
                MATCH (d:Document {path: $path})
                CREATE (c:Chunk {
                    text: $text,
                    index: $index,
                    embedding: $embedding
                })
                CREATE (c)-[:FROM_DOCUMENT]->(d)
            """, path=doc_path, text=chunk["text"], 
               index=chunk["index"], embedding=chunk["embedding"])
        print(f"Created {len(chunk_data)} Chunk nodes with embeddings")
        
        # Create NEXT_CHUNK relationships
        session.run("""
            MATCH (d:Document {path: $path})<-[:FROM_DOCUMENT]-(c:Chunk)
            WITH c ORDER BY c.index
            WITH collect(c) as chunks
            UNWIND range(0, size(chunks)-2) as i
            WITH chunks[i] as c1, chunks[i+1] as c2
            CREATE (c1)-[:NEXT_CHUNK]->(c2)
        """, path=doc_path)
        print("Created NEXT_CHUNK relationships")

store_chunks_with_embeddings(driver, DOCUMENT_PATH, chunk_embeddings)

## Create Vector Index

Create a vector index in Neo4j for efficient similarity search. The index uses cosine similarity to compare embeddings.

> **Note:** The vector dimensions must match your embedding model. Amazon Titan Text Embeddings V2 produces 1024-dimensional vectors. If you change `EMBEDDING_MODEL_ID` in `CONFIG.txt`, update the dimensions accordingly.

In [ ]:
INDEX_NAME = "chunkEmbeddings"

# Drop existing index if it exists
try:
    with driver.session() as session:
        session.run(f"DROP INDEX {INDEX_NAME} IF EXISTS")
        print(f"Dropped existing index: {INDEX_NAME}")
except Exception:
    pass

# Create new vector index (1024 dimensions for Titan V2)
create_vector_index(
    driver=driver,
    name=INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1024,
    similarity_fn="cosine"
)
print(f"Created vector index: {INDEX_NAME}")

## Vector Similarity Search

Now we can search for chunks that are semantically similar to a query. The search process:

1. **Embed the query** - Convert the search query to a vector using the same embedding model
2. **Find similar vectors** - Use the vector index to find chunks with similar embeddings
3. **Rank by score** - Return results ordered by cosine similarity (higher = more similar)

The Neo4j procedure `db.index.vector.queryNodes()` performs an efficient approximate nearest neighbor (ANN) search, which scales well even with millions of chunks.

In [ ]:
def vector_search(driver, embedder, query: str, top_k: int = 3):
    """Search for chunks similar to the query."""
    # Generate query embedding
    query_embedding = embedder.embed_query(query)
    
    with driver.session() as session:
        result = session.run("""
            CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
            YIELD node, score
            RETURN node.text as text, node.index as idx, score
            ORDER BY score DESC
        """, index_name=INDEX_NAME, top_k=top_k, embedding=query_embedding)
        
        return list(result)

# Test search
query = "What products does Apple make?"
print(f"Query: \"{query}\"\n")
print("=" * 60)

results = vector_search(driver, embedder, query)
for i, record in enumerate(results):
    print(f"\n[{i+1}] Score: {record['score']:.4f} (Chunk {record['idx']})")
    print(f"    {record['text']}")

# Vector Retriever

In the previous notebook, you performed vector search manually using Cypher queries. Now you'll use the **VectorRetriever** class from neo4j-graphrag, which abstracts away the complexity and provides a clean API for semantic search.

You'll also learn to use the **GraphRAG** class, which combines retrieval with LLM generation to build a complete question-answering pipeline.

**Prerequisites:** Complete [02 Embeddings](02_embeddings.ipynb) first to populate the graph with embeddings and create the vector index.

**Learning Objectives:**
- Use VectorRetriever for semantic search
- Inspect retrieval results and similarity scores
- Build a GraphRAG pipeline for question answering
- Understand how retrieved context improves LLM responses

## Initialize Vector Retriever

The `VectorRetriever` class handles all the complexity of semantic search:
- Automatically embeds your query using the provided embedder
- Queries the Neo4j vector index
- Returns results with similarity scores and content

This is much cleaner than writing manual Cypher queries for every search!

In [ ]:
# Initialize Vector Retriever
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

print("Vector Retriever initialized!")

**VectorRetriever Parameters:**
- `driver`: The Neo4j Python driver connection
- `index_name`: Name of the vector index to search (`chunkEmbeddings`)
- `embedder`: The embedding model to convert queries to vectors
- `return_properties`: List of node properties to include in results (e.g., `['text']`)

> **Tip:** You can add more properties to `return_properties` like `['text', 'index']` to get additional metadata about each chunk.

---

## Diagnostic Search

Before building the full RAG pipeline, it's useful to inspect raw retrieval results. This helps you verify:
- The vector index is working correctly
- The right chunks are being retrieved for your queries
- Similarity scores are reasonable (higher is better, typically 0.7+ indicates good relevance)

In [ ]:
# Simple Vector Search
query = "What products does Apple make?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(result.items)}\n")
for item in result.items:
    score = item.metadata.get('score', 'N/A')
    node_id = item.metadata.get('id', 'N/A')
    content_preview = str(item.content)[:100]
    print(f"Score: {score:.4f}, Content: {content_preview}..., id: {node_id}")

**How it works:**  
1. The example `query`, "What products does Apple make?", is created
2. `vector_retriever.search()` runs the query and returns the top 5 matches based on vector similarity.
3. The results are formatted displaying:
    * The similarity score (`Score`)
    * A snippet of the retrieved content (`Content`)
    * The unique identifier for each chunk (`id`)

This diagnostic helps you verify that the vector search is working and inspect the quality of the top results for your query.

> **Tip:**
> Inspecting the returned results to verify relevance can help you to adjust your chunking or embedding strategy.

In [ ]:
# Initialize LLM and Embedder from AWS Bedrock
llm = get_llm()
#embedder = get_embedder()

#print(f"LLM: {llm.model_id}")
#print(f"Embedder: {embedder.model_id}")

## Graph Retrieval-Augmented Generation (GraphRAG)

Now let's combine retrieval with generation. The `GraphRAG` class orchestrates a complete RAG pipeline:

1. **Retrieve** - Use the VectorRetriever to find relevant chunks
2. **Augment** - Format the retrieved chunks as context for the LLM
3. **Generate** - Send the query + context to the LLM for a grounded answer

This is the core pattern of RAG: instead of asking the LLM to answer from its training data alone, we provide relevant context from our knowledge graph so the answer is grounded in actual data.

In [ ]:
# Initialize GraphRAG and Perform Search
query = "What products does Apple make?"
rag = GraphRAG(
    llm=llm,
    retriever=vector_retriever
)
response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

**How the GraphRAG Pipeline Works:**

1. **Query received**: "What products does Apple make?"
2. **Retrieval**: The VectorRetriever finds the top-k most similar chunks
3. **Context formatting**: The retrieved chunks are formatted into a prompt
4. **LLM generation**: Claude receives both the question and the context
5. **Response**: The LLM generates an answer grounded in the retrieved data

**Key parameters:**
- `retriever_config={"top_k": 5}`: Retrieve 5 chunks to use as context
- `return_context=True`: Include the retrieved chunks in the response (useful for debugging)

The answer is now **grounded** in your actual SEC filing data rather than the LLM's general knowledge!

## Try Different Queries

Experiment with the vector retriever by modifying the `query`.

In [ ]:
# Try different queries
queries = [
    "What services does Apple offer?",
    "When does Apple's fiscal year end?",
    "Tell me about Apple's product line"
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    response = rag.search(query, retriever_config={"top_k": 3})
    print(f"Answer: {response.answer}")

## Summary

In this notebook, you built your first complete GraphRAG pipeline:

1. **VectorRetriever** - Abstracts vector search into a simple `search()` method. No more manual Cypher queries for embedding lookups.

2. **Diagnostic inspection** - Viewing raw retrieval results helps debug and tune your chunking/embedding strategy.

3. **GraphRAG pipeline** - Combines retrieval + LLM generation for grounded question answering. The LLM's response is based on actual data from your knowledge graph.

**Current limitation:** The VectorRetriever only returns the matched chunks themselves. But what if a question requires context that spans multiple chunks? In the next notebook, you'll learn to use **VectorCypherRetriever** to traverse graph relationships and include adjacent chunks for expanded context.

---

**Next:** [Vector Cypher Retriever](04_vector_cypher_retriever.ipynb)

## VectorCypherRetriever with Custom Query

The `VectorCypherRetriever` adds a key capability: after finding chunks via vector search, it runs a **custom Cypher query** starting from those matched chunks. This lets you:

- Traverse to the source document for provenance
- Follow `NEXT_CHUNK` relationships to get surrounding context
- Include related entities extracted from the text
- Gather any other graph data connected to the matched chunks

The `node` variable in your Cypher query refers to each chunk returned by the vector search.

In [ ]:
# Custom Cypher query that returns chunk context with document info and adjacent chunks
context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
RETURN 
    node.text AS context,
    doc.path AS document,
    node.index AS chunk_index,
    prev.text AS previous_chunk,
    next.text AS next_chunk
"""

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=context_query
)

print("VectorCypherRetriever initialized!")

**Understanding the Retrieval Query:**

```cypher
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)      -- Find the source document
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)  -- Find the previous chunk (if any)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)  -- Find the next chunk (if any)
RETURN ...
```

The query returns:
1. `node.text` - The matched chunk's text (what vector search found)
2. `doc.path` - Which document this chunk came from
3. `node.index` - The chunk's position in the document
4. `prev.text` - The text that comes *before* this chunk
5. `next.text` - The text that comes *after* this chunk

> **Why OPTIONAL MATCH?** The first and last chunks in a document don't have previous/next neighbors. OPTIONAL MATCH returns NULL instead of failing.

Now let's use this retriever in a GraphRAG pipeline:

In [ ]:
# Initialize GraphRAG and Perform Search
query = "What products and services does Apple offer?"

rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)
response = rag.search(query, retriever_config={"top_k": 3}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

## Inspecting Retrieved Context

One of the best ways to debug and improve your RAG system is to inspect what context is actually being passed to the LLM. Let's look at what the VectorCypherRetriever returned:

In [ ]:
# View the context used in this query
print("Retrieved Context:")
print("=" * 60)
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[Result {i+1}]")
    print(item.content)

## Expanded Context Window Pattern

The previous query returned separate fields for prev/current/next chunks. A more powerful pattern is to **concatenate them into a single expanded context string**. This gives the LLM a larger window of continuous text to work with.

Think of it like this: if your chunks are 400 characters each, this pattern gives the LLM ~1200 characters of context (prev + current + next) for each vector match, while only using the chunk size for embedding accuracy.

In [ ]:
# Query that combines current chunk with adjacent chunks for expanded context
expanded_context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
WITH node, doc, prev, next
RETURN 
    COALESCE(prev.text + ' ', '') + node.text + COALESCE(' ' + next.text, '') AS expanded_context,
    doc.path AS source_document,
    node.index AS center_chunk_index
"""

expanded_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=expanded_context_query
)

# Test with expanded context
query = "Tell me about Apple's services"
rag_expanded = GraphRAG(llm=llm, retriever=expanded_retriever)
response = rag_expanded.search(query, retriever_config={"top_k": 2}, return_context=True)

print(f"Query: \"{query}\"\n")
print("Answer:")
print(response.answer)

In [ ]:
# View the expanded context
print("\nExpanded Context (includes adjacent chunks):")
print("=" * 60)
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[Result {i+1}]")
    print(item.content)

## Comparing Standard vs Expanded Context

Let's see the difference in practice. We'll ask the same question using:
1. **Standard VectorRetriever** - Returns only the matched chunks
2. **VectorCypherRetriever with expanded context** - Returns matched chunks + neighbors

Notice how the expanded context often produces more complete, nuanced answers because the LLM has more information to work with.

In [ ]:
# Standard retriever (no graph traversal)
standard_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

query = "What does Apple do?"

# Standard retriever
print("=== Standard VectorRetriever ===")
rag_standard = GraphRAG(llm=llm, retriever=standard_retriever)
response_standard = rag_standard.search(query, retriever_config={"top_k": 2})
print(response_standard.answer)

# Expanded context retriever
print("\n=== VectorCypherRetriever (Expanded Context) ===")
response_expanded = rag_expanded.search(query, retriever_config={"top_k": 2})
print(response_expanded.answer)

## Summary

In this notebook, you learned the most powerful retrieval pattern in GraphRAG:

1. **VectorCypherRetriever** - Combines vector search with custom Cypher queries. The `node` variable in your query represents each chunk found by vector search.

2. **Custom retrieval queries** - You can traverse any graph relationships from matched chunks: documents, adjacent chunks, extracted entities, or any other connected data.

3. **Expanded context windows** - By concatenating adjacent chunks, you give the LLM more context while maintaining precise vector search. This often dramatically improves answer quality.

4. **The power of graphs** - This is what separates GraphRAG from simple vector stores. The relationships in your graph (NEXT_CHUNK, FROM_DOCUMENT) enable retrieval patterns that aren't possible with vectors alone.

**Key takeaway:** Vector search finds the needle in the haystack. Graph traversal provides the context around the needle so the LLM understands what it found.

---

**Congratulations!** You've completed the GraphRAG labs. You now know how to:
- Structure documents as graphs (Document → Chunk with NEXT_CHUNK chains)
- Create and search vector embeddings in Neo4j
- Build GraphRAG pipelines with VectorRetriever
- Enhance retrieval with custom Cypher queries

Continue to [Lab 6 - Aura Agents API](../Lab_6_Aura_Agents_API/README.md) to learn how to call Aura Agents programmatically.

In [ ]:
# Cleanup
neo4j.close()